# 06. 테스트데이터 예측결과 파일 생성과 재현성 점검

평가 항목 6(동일 환경에서 전처리부터 학습·추론·결과생성까지 자동 실행)의 마지막 단계입니다. 앞 노트북들이 저장한 예측을 모아 **제출용 예측결과 파일** 하나를 만들고, 파일에서 다시 계산한 지표가 앞 노트북의 표와 같은지 확인합니다.

| 파일 | 내용 |
|---|---|
| `outputs/predictions/test_predictions.csv` | **제출용.** 평가 기간(8/9~9/14) 시간별: 실제 시간 최대, 1시간 앞·하루 앞 예측(15분 모델·시간 최대 모델), 예측구간, 피크 위험 확률, 경보 |
| `outputs/predictions/final_hour_ahead.csv`, `final_day_ahead.csv` | 15분 모델의 15분 단위 최종 예측(02-2) |

> 예측은 모두 '그 시점 이전 자료로만 학습한 모델'의 결과입니다(8/9~9/14를 1주씩, 매주 그 이전 자료로 학습). 피크 위험 확률의 보정과 경보 문턱은 앞 주로 정하므로 2~5주차(8/16~)에만 값이 있습니다.

In [1]:
# [코드 R-01] 준비: 입력 데이터 확인(SHA-256)과 실행 환경
import hashlib
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from IPython.display import display
import hourly_utils as mu

EXPECTED = {'data/raw/okm_augumented_2021.csv': '8f7af2e49366c93e1d6f5fdef4b5e350066c1792ac463c2c2886e370f4674830'}
for rel, sha in EXPECTED.items():
    got = hashlib.sha256((mu.ROOT / rel).read_bytes()).hexdigest()
    assert got == sha, f'{rel}의 내용이 다릅니다'
    print(f'{rel}: SHA-256 일치')
env = mu.pu.get_env_info()
print(f"Python {env['python']} | {env['platform']} | CPU {env['cpu']} | 스레드 {env['n_threads']}")
print('주요 패키지:', ', '.join(f'{k} {v}' for k, v in env['packages'].items() if v))

data/raw/okm_augumented_2021.csv: SHA-256 일치


Python 3.12.14 | Windows-11-10.0.26200-SP0 | CPU 11th Gen Intel(R) Core(TM) i5-1135G7 @ 2.40GHz | 스레드 4
주요 패키지: numpy 2.4.6, pandas 2.3.3, scikit-learn 1.9.1, lightgbm 4.7.0, xgboost 3.2.0, catboost 1.2.10, statsforecast 2.1.1, neuralforecast 3.2.2, optuna 5.0.0, shap 0.51.0, pyarrow 25.0.1, matplotlib 3.11.2, seaborn 0.13.2, torch 2.14.1, keras 3.15.1, chronos-forecasting 2.3.2, papermill 2.7.0, nbconvert 7.17.1


## 1. 예측결과 파일 만들기

열 이름은 한글로 쓰고 단위(kW)를 붙였습니다.

| 열 | 뜻 | 출처 |
|---|---|---|
| 실제_시간최대_kW, 정답사용, 피크 | 그 시간 15분 4칸 중 최댓값, 4칸 모두 정답으로 쓸 수 있는지, 188.7kW 이상인지 | 01-2 |
| 1시간앞_15분모델_kW | 15분 모델 최종 예측(선형 회귀 + Chronos-2)의 4칸 최대 | 02-2 |
| 1시간앞_시간최대_q10·q50·q90_kW | 시간 최대 Chronos-2 zero-shot의 10%·50%·90% 분위수 | 02-4 |
| 1시간앞_피크확률_분위수 | q10~q90로 계산한 P(시간 최대 ≥ 188.7) | 02-4 |
| 1시간앞_피크확률_보정 | 위 확률을 앞 주로 isotonic 보정한 값(2~5주차) | 02-4 |
| 경보_기본, 경보_재현율우선 | 15분 모델 예측 ≥ 178.7kW / 시간 최대 q50 ≥ 178.7kW | 02-3·02-4 |
| 하루앞_15분모델_kW, 하루앞_시간최대_q10·q50·q90_kW | 15분 기준 곡선의 4칸 최대 / 시간 최대 Chronos-2 + 공변량 분위수 | 02-2·02-4 |

In [2]:
# [코드 R-02] 제출용 예측결과 파일 만들기
P = mu.OUT / 'preds'
pts = pd.read_parquet(P / 'h_point_preds.parquet')
ca, cb = pd.read_parquet(P / 'chronos_A.parquet'), pd.read_parquet(P / 'chronos_B.parquet')
R = pd.read_parquet(P / 'h_probs_A.parquet')
idx = pts.index
out = pd.DataFrame(index=idx)
out.index.name = '시각'
out['주차'] = pts['week']
out['하루종류'] = pts['하루종류']
out['실제_시간최대_kW'] = pts['y']
out['정답사용'] = pts['valid'].astype(bool)
out['피크'] = (pts['y'] >= mu.THETA) & out['정답사용']
out['1시간앞_15분모델_kW'] = pts['1시간 앞|15분 최종: 선형 회귀 + Chronos-2']
for q in ['q10', 'q50', 'q90']:
    out[f'1시간앞_시간최대_{q}_kW'] = ca[q].reindex(idx)
out['1시간앞_피크확률_분위수'] = pd.Series(mu.pu.peak_prob_from_quantiles(ca, mu.THETA), index=ca.index).reindex(idx)
out['1시간앞_피크확률_보정'] = R['p_chr|p'].reindex(idx)
out['경보_기본'] = (out['1시간앞_15분모델_kW'] >= mu.READY).where(out['1시간앞_15분모델_kW'].notna())
out['경보_재현율우선'] = out['1시간앞_시간최대_q50_kW'] >= mu.READY
out['하루앞_15분모델_kW'] = pts['하루 앞|15분 최종: 기준 곡선']
for q in ['q10', 'q50', 'q90']:
    out[f'하루앞_시간최대_{q}_kW'] = cb[q].reindex(idx)
path = mu.OUT / 'predictions' / 'test_predictions.csv'
out.round(4).to_csv(path, encoding='utf-8-sig')
print(f'저장: {path.relative_to(mu.ROOT)} — {len(out)}행 × {out.shape[1]}열, '
      f'{out.index.min():%Y-%m-%d %H:%M} ~ {out.index.max():%Y-%m-%d %H:%M}')
display(out.head(3))

(공개 저장소에서는 원자료 행이 보이는 출력을 지웠습니다. 실행하면 다시 나타납니다.)


## 2. 파일에서 다시 계산한 지표가 앞 노트북과 같은지 확인

저장한 파일만 다시 읽어 MAE와 경보 지표를 계산하고, 02-4의 표(H-01·H-04)와 같은지 `assert`로 확인합니다. 같으면 이 파일이 보고한 결과를 그대로 담고 있다는 뜻입니다.

In [3]:
# [코드 R-03] 파일 기반 재계산과 앞 노트북 표 대조 [표 R-01]
f = pd.read_csv(path, encoding='utf-8-sig', parse_dates=['시각'], index_col='시각')
ok = f['정답사용'] & f['1시간앞_15분모델_kW'].notna()
y = f.loc[ok, '실제_시간최대_kW']
rows = [{'항목': '1시간 앞 MAE — 15분 모델', '값': (f.loc[ok, '1시간앞_15분모델_kW'] - y).abs().mean()},
        {'항목': '1시간 앞 MAE — 시간 최대 Chronos-2', '값': (f.loc[ok, '1시간앞_시간최대_q50_kW'] - y).abs().mean()},
        {'항목': '하루 앞 MAE — 15분 기준 곡선', '값': (f.loc[ok, '하루앞_15분모델_kW'] - y).abs().mean()},
        {'항목': '하루 앞 MAE — 시간 최대 Chronos-2 + 공변량', '값': (f.loc[ok, '하루앞_시간최대_q50_kW'] - y).abs().mean()}]
ev = ok & f['주차'].isin(mu.EVAL_WEEKS)
for col, name in [('경보_기본', '15분 모델: 점예측 ≥ 178.7'), ('경보_재현율우선', '시간 최대 Chronos-2 ≥ 178.7')]:
    r = mu.cls_row(f.loc[ev, '피크'].to_numpy(), f.loc[ev, col].astype(bool).to_numpy())
    rows += [{'항목': f'{name} — 재현율', '값': r['재현율']}, {'항목': f'{name} — 정밀도', '값': r['정밀도']},
             {'항목': f'{name} — F1', '값': r['F1']}]
R01 = pd.DataFrame(rows)
h04 = pd.read_csv(mu.OUT / 'tables' / 'h04_point_mae.csv', encoding='utf-8-sig').set_index(['예측', '모델'])['MAE']
h01 = pd.read_csv(mu.OUT / 'tables' / 'h01_hour_alarm.csv', encoding='utf-8-sig').set_index('방법')
ref = [h04[('1시간 앞', '15분 최종: 선형 회귀 + Chronos-2')], h04[('1시간 앞', '시간 최대: Chronos-2 zero-shot')],
       h04[('하루 앞', '15분 최종: 기준 곡선')], h04[('하루 앞', '시간 최대: Chronos-2 + 공변량')]]
for name in ['15분 모델: 점예측 ≥ 178.7', '시간 최대 Chronos-2 ≥ 178.7']:
    ref += [h01.loc[name, '재현율'], h01.loc[name, '정밀도'], h01.loc[name, 'F1']]
R01['앞 노트북 표'] = ref
assert np.allclose(R01['값'], R01['앞 노트북 표'], atol=1e-3), '파일과 앞 노트북 표가 다릅니다'
mu.save_table(R01, 'r01_file_check')
display(R01.round(3))
print('파일에서 다시 계산한 값이 02-4의 표와 모두 같습니다(허용 오차 0.001).')

,항목,값,앞 노트북 표
0,1시간 앞 MAE — 15분 모델,5.853,5.853
1,1시간 앞 MAE — 시간 최대 Chronos-2,5.780,5.780
2,하루 앞 MAE — 15분 기준 곡선,7.064,7.064
3,하루 앞 MAE — 시간 최대 Chronos-2 + 공변량,6.891,6.891
4,15분 모델: 점예측 ≥ 178.7 — 재현율,0.833,0.833
5,15분 모델: 점예측 ≥ 178.7 — 정밀도,0.372,0.372
6,15분 모델: 점예측 ≥ 178.7 — F1,0.515,0.515
7,시간 최대 Chronos-2 ≥ 178.7 — 재현율,0.976,0.976
8,시간 최대 Chronos-2 ≥ 178.7 — 정밀도,0.328,0.328
9,시간 최대 Chronos-2 ≥ 178.7 — F1,0.491,0.491


파일에서 다시 계산한 값이 02-4의 표와 모두 같습니다(허용 오차 0.001).
